# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibily one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probabilty of net winning or lossing after 50 rounds?


9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 

In [1]:
class Card:
    __suits = ["Clubs", "Diamonds", "Hearts", "Spades", "ShuffleCard"]
    __values = list(range(2, 11)) + ["Jack", "Queen", "King", "Ace"]

    def __init__(self, suit, value=None):
        if suit not in self.__suits:
            raise ValueError(f"Error, bad suit: {suit}")
        if value not in self.__values and suit != "ShuffleCard":
            raise ValueError(f"Error, bad value: {value}")

        self.__suit = suit
        self.__value = value

    def value(self):
        return self.__value

    def suit(self):
        return self.__suit

    def numerical_value(self):
        """Returns numerical value of the card. Aces default to 1."""
        if self.__value == "Ace":
            return 1
        elif self.__value in ["Jack", "Queen", "King"]:
            return 10
        else:
            return self.__value

    def shuffle_card(self):
        return self.__suit == "ShuffleCard"

    def __str__(self):
        if self.shuffle_card():
            return "Shuffle Card"
        else:
            return f"{self.__value} of {self.__suit}"

    __repr__ = __str__


In [2]:
import random

class Deck():
    __suits = ["Clubs", "Diamonds", "Hearts", "Spades"]
    __values = list(range(2,11)) + [ "Jack", "Queen", "King", "Ace"]

    def __init__(self, n_decks=6):
        self.__n_decks = n_decks
        self.__cards = self.__generate_deck()
        self.shuffle()

    def __generate_deck(self):
        deck = [Card(suit, value) for suit in self.__suits for value in self.__values] * self.__n_decks

        # Add the shuffle card (plastic card) randomly between 75-85% into the deck
        shuffle_position = random.randint(int(len(deck) * 0.75), int(len(deck) * 0.85))
        deck.insert(shuffle_position, Card("ShuffleCard"))

        return deck

    def shuffle(self):
        """Shuffle the deck"""
        random.shuffle(self.__cards)

    def deal(self):
        """Deals a card. If the deck is empty, it regenerates and shuffles before dealing."""
        if not self.__cards:  # Check if the deck is empty
            self.__cards = self.__generate_deck()
            self.shuffle()

        return self.__cards.pop()


In [3]:
class Hand:
    def __init__(self):
        self.__cards = []

    def add_card(self, card):
        self.__cards.append(card)

    def clear_hand(self):
        self.__cards = []

    def hand_value(self):
        card_values = [card.numerical_value() for card in self.__cards]

        n_aces = card_values.count(1)  # Count Aces
        hand_total = sum(card_values)

        if n_aces == 0:
            return hand_total  # No Aces, return normal sum

        # Handle Aces (Ace can be 1 or 11)
        ace_as_one = hand_total
        ace_as_eleven = hand_total + 10  # One Ace is counted as 11

        if ace_as_eleven <= 21:
            return ace_as_eleven
        return ace_as_one  # If 11 makes us bust, Ace stays as 1

    def has_blackjack(self):
        """Check if the hand is a blackjack (Ace + 10-value card)."""
        return len(self.__cards) == 2 and self.hand_value() == 21

    def is_busted(self):
        """Check if the hand is over 21 (busted)."""
        return self.hand_value() > 21

    def __str__(self):
        return ", ".join(str(card) for card in self.__cards)

    def show_hand(self, hide_first_card=False):
        """Show the hand, optionally hiding the first card (for dealer)."""
        if hide_first_card:
            return "[Hidden], " + ", ".join(str(card) for card in self.__cards[1:])
        return str(self)


In [4]:
class PlayerBase:
    def __init__(self, name, n_chips=100):
        self.__name = name
        self.__n_chips = n_chips
        self.hand = Hand()  

    def name(self):
        return self.__name

    def chips(self):
        return self.__n_chips

    def pay(self, value=2):
        self.__n_chips += value

    def deduct(self, value=2):
        self.__n_chips = max(0, self.__n_chips - value)

    def reset_hand(self):
        self.hand = Hand()

    def play_hand(self):
        raise NotImplementedError("Subclasses must implement `play_hand()`.")

    def __str__(self):
        return f"{self.__name} ({self.__n_chips} chips)"

    __repr__ = __str__

class Dealer(PlayerBase):
    def __init__(self, threshold=17):
        super().__init__("Mr. Dealer", 1000)
        self.__threshold = threshold

    def play_hand(self, deck):
        while self.hand_value() < self.__threshold:
            self.add_card_to_hand(deck.deal())  
        return False  

class ConsolePlayer(PlayerBase):
    def play_hand(self):
        while True:
            action = input(f"{self.name()}, your hand: {self.hand}. Hit (Y/N)? ").strip().upper()
            if action == "Y":
                return True  
            elif action == "N":
                return False  
            else:
                print("Invalid input. Please enter 'Y' or 'N'.")

class AIPlayer(PlayerBase):
    def __init__(self, name, n_chips=100, hit_threshold=16):
        super().__init__(name, n_chips)
        self.hit_threshold = hit_threshold

    def play_hand(self):
        return self.hand.hand_value() < self.hit_threshold  


class RandomPlayer(PlayerBase):
    
    def play_hand(self):
        return random.choice([True, False])  


class ConservativePlayer(PlayerBase):
    
    def play_hand(self):
        return self.hand.hand_value() < 13


In [18]:
import random

class Game:
    def __init__(self, n_decks=6, verbose=True):
        self.deck = Deck(n_decks)
        self.players = []
        self.all_players = []
        self.shuffle = False
        self.dealer = None  
        self.verbose = verbose  

    def add_player(self, player):
        self.players.append(player)
        self.all_players.append(player)

    def deal_and_check_shuffle(self):
        card = self.deck.deal()
        if card.shuffle_card():
            if self.verbose:
                print("\n[SHUFFLE TRIGGERED] Reshuffling deck...\n")
            self.deck.shuffle()
            card = self.deck.deal()
        return card

    def reset_hands(self):
        for player in self.players:
            player.reset_hand()
        self.dealer.reset_hand()

    def deal_initial_cards(self):
        for _ in range(2):
            for player in self.players:
                player.hand.add_card(self.deal_and_check_shuffle())

    def player_turn(self, player):
        while player.hand.hand_value() < 21:
            action = player.play_hand()  
            if action:
                player.hand.add_card(self.deal_and_check_shuffle())
            else:
                break

    def dealer_turn(self):
        while self.dealer.hand.hand_value() < 17:
            self.dealer.hand.add_card(self.deal_and_check_shuffle())

    def determine_winners(self):
        dealer_value = self.dealer.hand.hand_value()
        if self.verbose:
            print("\n--- Round Results ---")
            print(f"Dealer's hand: {self.dealer.hand} ({dealer_value})")

        for player in self.players:
            player_value = player.hand.hand_value()
            if self.verbose:
                print(f"{player.name()}: {player.hand} ({player_value})", end=" - ")

            if player.hand.is_busted():
                if self.verbose:
                    print("Busted! Lose 2 chips.")
                player.deduct(2)
            elif dealer_value > 21:
                if self.verbose:
                    print("Dealer Busted! Win 2 chips.")
                player.pay(2)
            elif player_value > dealer_value:
                if self.verbose:
                    print("Win! Win 2 chips.")
                player.pay(2)
            elif player_value == 21:
                if self.verbose:
                    print("Blackjack! Win 3 chips.")
                player.pay(3)
            else:
                if self.verbose:
                    print("Lose! Lose 2 chips.")
                player.deduct(2)

    def play_round(self):
        if self.verbose:
            print("\n--- New Round ---")
        self.reset_hands()
        self.deal_initial_cards()

        for player in self.players:
            if not isinstance(player, Dealer):
                if self.verbose:
                    print(f"\n{player.name()}'s turn:")
                self.player_turn(player)

        if self.verbose:
            print("\nDealer's turn:")
        self.dealer_turn()

        self.determine_winners()

    def play_game(self, n_rounds=1):
        if self.dealer is None:
            self.dealer = Dealer()
            self.add_player(self.dealer)

        for _ in range(n_rounds):
            self.play_round()
            if self.verbose:
                input("\nPress Enter to continue to the next round...")


In [16]:
my_game = Game()
my_game.add_player(ConsolePlayer("Brian", 100))

In [17]:
my_game.play_game(3)


--- New Round ---

Brian's turn:
Brian, your hand: 6 of Diamonds, 8 of Hearts. Hit (Y/N)? y
Brian, your hand: 6 of Diamonds, 8 of Hearts, 6 of Hearts. Hit (Y/N)? n

Dealer's turn:

--- Round Results ---
Dealer's hand: Queen of Clubs, Queen of Clubs (20)
Brian: 6 of Diamonds, 8 of Hearts, 6 of Hearts (20) - Lose! Lose 2 chips.
Mr. Dealer: Queen of Clubs, Queen of Clubs (20) - Lose! Lose 2 chips.

Press Enter to continue to the next round...

--- New Round ---

Brian's turn:
Brian, your hand: 4 of Hearts, 8 of Spades. Hit (Y/N)? y
Brian, your hand: 4 of Hearts, 8 of Spades, 7 of Diamonds. Hit (Y/N)? 
Invalid input. Please enter 'Y' or 'N'.
Brian, your hand: 4 of Hearts, 8 of Spades, 7 of Diamonds. Hit (Y/N)? n

Dealer's turn:

--- Round Results ---
Dealer's hand: Queen of Hearts, 2 of Clubs, 10 of Spades (22)
Brian: 4 of Hearts, 8 of Spades, 7 of Diamonds (19) - Dealer Busted! Win 2 chips.
Mr. Dealer: Queen of Hearts, 2 of Clubs, 10 of Spades (22) - Busted! Lose 2 chips.

Press Enter to

In [14]:
game1 = Game(n_decks=6, verbose=False)


game1.add_player(AIPlayer("Aggressive AI", hit_threshold=18))  
game1.add_player(AIPlayer("Balanced AI", hit_threshold=16))  
game1.add_player(ConservativePlayer("Conservative AI"))  
game1.add_player(RandomPlayer("Random AI"))  

game1.play_game(n_rounds=500)

for player in game1.players:
    print(f"{player.name()}: {player.chips()} chips")


Aggressive AI: 17 chips
Balanced AI: 2 chips
Conservative AI: 2 chips
Random AI: 4 chips
Mr. Dealer: 270 chips


In [20]:
import matplotlib.pyplot as plt

num_rounds = 500
game2 = Game(n_decks=6, verbose=False)

# Add AI players
game2.add_player(AIPlayer("Aggressive AI", hit_threshold=18))
game2.add_player(AIPlayer("Balanced AI", hit_threshold=16))
game2.add_player(ConservativePlayer("Conservative AI"))
game2.add_player(RandomPlayer("Random AI"))
game2.add_player(Dealer())  # Dealer

chip_history = {player.name(): [] for player in game2.players}

# Run the game and track chip counts
for _ in range(num_rounds):
    game2.play_round()
    for player in game2.players:
        chip_history[player.name()].append(player.chips())

# Plot results
for name, chips in chip_history.items():
    plt.plot(range(num_rounds), chips, label=name)

plt.xlabel("Rounds")
plt.ylabel("Chips")
plt.title("Player Performance Over Time")
plt.legend()
plt.show()


AttributeError: 'NoneType' object has no attribute 'reset_hand'